# **Basic Tasks**

In [0]:
orders_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/cyntexa_dev/sales/sales_volume/orders.csv")

In [0]:
orders_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

In [0]:
null_counts = orders_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in orders_df.columns
])

In [0]:
null_counts.display()

I have null containing column is Customer Email, Region, total amount.

In [0]:
missing_email_df = orders_df.filter(col("Customer Email").isNull())
missing_region_df = orders_df.filter(col("Region").isNull())
missing_total_amount_df = orders_df.filter(col("Total Amount").isNull())

In [0]:
missing_email_df.display()

In [0]:
missing_region_df.display()

In [0]:
missing_total_amount_df.display()

In [0]:
print("Original row count:", orders_df.count())

In [0]:
distinct_df = orders_df.distinct()

print("Distinct row count:", distinct_df.count())

In [0]:
deduplicated_df = orders_df.dropDuplicates()

print("Original row count:", orders_df.count())
print("After dropDuplicates:", deduplicated_df.count())

In [0]:
from pyspark.sql.functions import count

duplicate_order_ids = (
    orders_df
    .groupBy("Order ID")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

## Task 1 — Data Quality Assessment

The e-commerce orders CSV was loaded into a PySpark DataFrame. The dataset contains several data-quality issues, including NULL values and duplicate records.

The `filter()` operation was used to identify records containing NULL values in important columns such as Customer Email, Region, and Total Amount.

The `distinct()` operation was used to determine the number of unique rows, while `dropDuplicates()` was used to remove duplicate records.

Duplicate Order IDs were also identified by grouping the data by Order ID and filtering groups with a count greater than one. This helps identify potential duplicate business records before further transformations are performed.

In [0]:
orders_renamed_df = (
    orders_renamed_df
    .withColumnRenamed("Order ID","order_id")
    .withColumnRenamed("Customer ID","customer_id")
    .withColumnRenamed("Customer Name","customer_name")
    .withColumnRenamed("Product Category","product_category")
    .withColumnRenamed("Customer Email", "customer_email")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("Region", "region")
    .withColumnRenamed("Product Name", "product_name")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("Unit Price", "unit_price")
    .withColumnRenamed("Total Amount", "total_amount")
    .withColumnRenamed("Order Date", "order_date")
    .withColumnRenamed("Order Status", "order_status")
)

In [0]:
orders_renamed_df.printSchema()

## Task 2 — Column Renaming

The `withColumnRenamed()` function was used to rename the columns into consistent, lowercase, snake_case names. This improves readability and makes the columns easier to reference in PySpark transformations.

The columns `Order ID`, `Customer ID`, `Customer Name`, and `Product Category` were renamed to `order_id`, `customer_id`, `customer_name`, and `product_category`.

The columns were standardized using lowercase snake_case naming conventions to make the DataFrame easier to use in subsequent PySpark transformations.

# **Intermediate Tasks**

In [0]:
from pyspark.sql.functions import (col, trim, lower, upper, when, to_date, regexp_replace)
from pyspark.sql.functions import initcap

In [0]:
def chained_DataFrame_transformation_pipeline(orders_renamed_df):

    # Remove exact duplicate rows
    clean_orders_df = orders_renamed_df.dropDuplicates()

    # Clean whitespace
    clean_orders_df = (
    clean_orders_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("customer_email", trim(col("customer_email")))
    .withColumn("city", trim(col("city")))
    .withColumn("region", trim(col("region")))
    .withColumn("product_category", trim(col("product_category")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("order_status", trim(col("order_status")))
    )

    # Standardize casing
    clean_orders_df = (
    clean_orders_df
    .withColumn("customer_name", initcap(col("customer_name")))
    .withColumn("city", initcap(col("city")))
    .withColumn("region", initcap(col("region")))
    .withColumn("product_category", initcap(col("product_category")))
    .withColumn("product_name", initcap(col("product_name")))
    .withColumn("order_status", initcap(col("order_status")))
    )
    # For email, we want lowercase
    clean_orders_df = clean_orders_df.withColumn(
    "customer_email",
    lower(col("customer_email"))
    )
    
    # Handle NULL values

    # For missing Customer Email
    clean_orders_df = clean_orders_df.withColumn(
    "customer_email",
    when(
        col("customer_email").isNull(),
        "unknown@example.com"
    ).otherwise(col("customer_email"))
    )

    # For missing regions
    clean_orders_df = clean_orders_df.withColumn(
        "region",
        when(
            col("region").isNull(),
                "unknown"
        ).otherwise(col("region")
        )
    )
    
    # For missing city
    clean_orders_df = clean_orders_df.withColumn(
        "city",
        when(
            col("city").isNull(),
            "unknown"
        ).otherwise(col("city"))
    )

    # For missing quantity
    clean_orders_df = clean_orders_df.withColumn(
        "quantity",
        when(
            col("quantity").isNull(),
            1
        ).otherwise(col("quantity"))
        )
    
    # remove unwanted characters from total amount
    clean_orders_df = clean_orders_df.withColumn(
    "total_amount",
    regexp_replace(
        col("total_amount").cast("string"),
        "[$,]",
        ""
    ).cast("double")
    )

    # Handle missing total_amount
    clean_orders_df = clean_orders_df.withColumn(
        "total_amount",
        when(
            col("total_amount").isNull(),
            col("quantity") * col("unit_price")
        ).otherwise(col("total_amount"))
    )
    
    # Convert order_date - handle multiple formats
    from pyspark.sql.functions import coalesce, try_to_date
    clean_orders_df = clean_orders_df.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "MM-dd-yyyy"),
        try_to_date(col("order_date"), "dd/MM/yyyy"),
        try_to_date(col("order_date"), "MM/dd/yyyy"),
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd-MM-yyyy")
    )
    )

    # Remove duplicate Order IDs
    clean_orders_df = clean_orders_df.dropDuplicates(
    ["order_id"]
    )
    
    return clean_orders_df

In [0]:
clean_orders_df = chained_DataFrame_transformation_pipeline(orders_renamed_df)

## Task 4 — Complete Data Cleaning Pipeline

A chained PySpark DataFrame transformation pipeline was created to clean the e-commerce orders dataset.

The pipeline removes duplicate records, trims unnecessary whitespace, standardizes text casing, handles NULL values, cleans and converts the total amount to a numeric type, converts the order date to DateType, and removes duplicate Order IDs.

The cleaned DataFrame was validated by checking remaining NULL values and duplicate Order IDs.

In [0]:
clean_orders_df.display()

In [0]:
category_reference = spark.createDataFrame(
    [
        ("Electronics", "Technology", "High Value"),
        ("Furniture", "Home & Office", "High Value"),
        ("Clothing", "Fashion", "Medium Value"),
        ("Grocery", "Daily Needs", "Low Value"),
        ("Books", "Education", "Low Value")
    ],
    [
        "product_category",
        "department",
        "category_segment"
    ]
)

In [0]:
category_reference.display()

In [0]:
from pyspark.sql.functions import countDistinct, sum, round

In [0]:
category_sales = (
    clean_orders_df
    .groupBy("product_category")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
)

In [0]:
category_sales.display()

In [0]:
category_report = (
    category_sales
    .join(
        category_reference,
        on="product_category",
        how="left"
    )
)

In [0]:
category_report.display()

In [0]:
category_report = category_report.withColumn(
    "average_order_value",
    round(
        col("total_revenue") / col("total_orders"),
        2
    )
)

In [0]:
final_category_report = (
    category_report
    .orderBy(col("total_revenue").desc())
)

In [0]:
unmatched_categories = (
    category_sales
    .join(
        category_reference,
        on="product_category",
        how="left_anti"
    )
)

display(unmatched_categories)

## Task 5 — Aggregation and Join

The cleaned orders DataFrame was aggregated by product category to calculate the total number of orders, total quantity sold, and total revenue.

A category reference DataFrame was created and joined with the aggregated sales data using a left join. The reference table provides additional business information such as department and category segment.

Average order value was also calculated using total revenue divided by the total number of orders. The final report was sorted by total revenue in descending order.

A left-anti join was used to identify categories that did not have a matching reference record.